## Load Dataset


In [2]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k")

data = dataset["train"]

print(data[0])

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


## Subset & Split
We use a 10,000-example subset (for practical training time on a single GPU) with an 80/20 train/test split. `seed=42` ensures reproducibility.

In [21]:
small = data.shuffle(seed=42).select(range(10000))

split = small.train_test_split(test_size=0.2)

train_data = split["train"]
test_data = split["test"]

## Format for Causal LM
We format each example into a single text string: `Instruction → Context (if present) → Answer`. This "completion-style" format is standard for causal language model fine-tuning — the model learns to generate the answer given the instruction prefix.

In [22]:
def format_example(ex):

    if ex["context"]:
        text = f"""Instruction: {ex['instruction']}

Context: {ex['context']}

Answer: {ex['response']}"""
    else:
        text = f"""Instruction: {ex['instruction']}

Answer: {ex['response']}"""

    return {"text": text}


train_data = train_data.map(format_example)
test_data = test_data.map(format_example)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [23]:
from transformers import pipeline 
ask_llm = pipeline(
    model= "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [25]:
print(ask_llm('User: Explain AI models and applications in healthcare. Assistant:'))

[{'generated_text': "User: Explain AI models and applications in healthcare. Assistant: Artificial intelligence (AI) refers to the application of computer science, data science, and machine learning to solve real-world problems. In healthcare, AI models are being used to improve diagnosis, treatment planning, and patient outcomes. One example of AI in healthcare is computer-assisted diagnosis (CAD) systems. CAD systems use machine learning algorithms to analyze medical images and identify the most relevant features that can help doctors make a diagnosis. AI can also be used to predict the likelihood of recurrence or complications after a patient undergoes a certain procedure, which can help optimize treatment plans and improve patient outcomes. Another example is smart hospital beds that use AI to monitor patients' vital signs and adjust the bed's position to ensure optimal medical conditions. This technology can help reduce hospital readmissions, optimize bed placement, and improve pa

## Load Model & Tokenizer

**Why TinyLlama-1.1B-Chat?**
- Small enough to fine-tune on a single consumer GPU (< 4GB VRAM with LoRA)
- Already instruction-tuned (chat variant), so it has a foundation for instruction-following
- 1.1B parameters demonstrates that LoRA can adapt even small models to specific tasks

We use `float16` precision to halve memory usage with minimal quality loss.

In [27]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)



In [28]:
import torch
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    torch_dtype=torch.float16
)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Tokenization

- **`max_length=256`**: Covers ~95% of Dolly examples (most instructions + responses fit in 256 tokens). Longer examples are truncated — a trade-off between memory and coverage.
- **`padding="max_length"`**: Fixed-length sequences enable efficient batching.
- **`labels = input_ids`**: For causal LM training, the target is the input shifted by one position — the model learns to predict the next token. The Trainer handles the shift internally.

In [30]:
def tokenize(example):
    tok = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tok["labels"] = tok["input_ids"].copy()  
    return tok

In [31]:
train_tok = train_data.map(tokenize, batched=True)
test_tok = test_data.map(tokenize, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


## Training Arguments

| Parameter | Value | Why |
|-----------|-------|-----|
| `per_device_train_batch_size=1` | Batch size | TinyLlama + LoRA fits in GPU memory with batch=1. Larger batches would need gradient accumulation. |
| `num_train_epochs=3` | Epochs | 3 passes over 8K examples = 24K gradient steps. Enough to learn the instruction format without overfitting. |
| `learning_rate=0.001` | Learning rate | Higher than typical full fine-tuning (1e-5) because LoRA only updates ~0.8M parameters. The small number of trainable weights can handle a larger LR without instability. |
| `fp16=True` | Mixed precision | Halves memory for activations and speeds up training on NVIDIA GPUs with Tensor Cores. |
| `save_steps=50` | Checkpoint frequency | Saves adapter weights every 50 steps for recovery if training is interrupted. |

In [32]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [33]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="lora-dolly",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    learning_rate=0.001,
    logging_steps=10,
    save_steps=50,
    fp16=True,
)


In [35]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok
)

In [36]:
trainer.train()

Step,Training Loss
10,7.005542
20,1.534701
30,1.134150
40,1.164587
50,1.159492
60,1.166079
70,1.253558
80,1.190888
90,1.217024
100,1.111658


TrainOutput(global_step=12000, training_loss=0.9618919397592545, metrics={'train_runtime': 7340.429, 'train_samples_per_second': 3.27, 'train_steps_per_second': 1.635, 'total_flos': 3.8177788133376e+16, 'train_loss': 0.9618919397592545, 'epoch': 3.0})

## Inference After Training

We generate answers using the fine-tuned model. Key generation parameters:
- **`temperature=0.7`**: Moderate randomness — less deterministic than greedy but avoids repetitive outputs
- **`top_p=0.9`**: Nucleus sampling — only considers tokens in the top 90% probability mass, filtering unlikely tokens
- **`max_new_tokens=150`**: Caps generation length to prevent runaway text

In [ ]:
import torch

def generate_answer(prompt, max_new_tokens=150):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )


    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    output = tokenizer.decode(out[0], skip_special_tokens=True)

    # remove prompt from output
    answer = output[len(prompt):].strip()

    return answer

In [49]:
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

In [50]:
preds = []
refs = []

for ex in test_data.select(range(20)):

    prompt = f"""
Instruction: {ex['instruction']}
Context: {ex['context']}
Answer:
"""

    out = generate_answer(prompt)

    preds.append(out)
    refs.append(ex["response"])

In [51]:
print(bleu.compute(predictions=preds, references=[[r] for r in refs]))
print(rouge.compute(predictions=preds, references=refs))

{'bleu': 0.07085090430099945, 'precisions': [0.28005115089514065, 0.11852331606217617, 0.08530183727034121, 0.07247340425531915], 'brevity_penalty': 0.5919715662753957, 'length_ratio': 0.6560402684563759, 'translation_length': 1564, 'reference_length': 2384}
{'rouge1': np.float64(0.3129926739252634), 'rouge2': np.float64(0.1499252403811412), 'rougeL': np.float64(0.23355925447228415), 'rougeLsum': np.float64(0.2461378672611912)}


## Save Adapter
We save only the LoRA adapter weights (~3MB), not the full 1.1B model. To reload, we load the base TinyLlama model and apply the adapter on top — this is done in `lora_eval.ipynb`.

In [56]:
from transformers import AutoModelForCausalLM, AutoTokenizer


model.save_pretrained('./model')
tokenizer.save_pretrained('./model')

('./model/tokenizer_config.json',
 './model/chat_template.jinja',
 './model/tokenizer.json')

## Next: Full Evaluation
Evaluation is handled in `lora_eval.ipynb` we load the saved adapter, run balanced sampling, and compare metrics against the DeepSeek baseline.